In [7]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import json
from tqdm import tqdm

In [18]:
import pandas as pd

# Đổi read_csv thành read_excel
file_path = 'Gia_phan_fixed.xlsx'
df = pd.read_excel(file_path)

# Hiển thị thử 5 dòng đầu để kiểm tra
print(df.head())

              Ngaydang                                                URL
0  2024-01-01 00:00:00  https://vietnambiz.vn/gia-phan-bon-hom-nay-11-...
1  2024-02-01 00:00:00  https://vietnambiz.vn/gia-phan-bon-hom-nay-21-...
2  2024-03-01 00:00:00  https://vietnambiz.vn/gia-phan-bon-hom-nay-31-...
3  2024-04-01 00:00:00  https://vietnambiz.vn/gia-phan-bon-hom-nay-41-...
4  2024-05-01 00:00:00  https://vietnambiz.vn/gia-phan-bon-hom-nay-51-...


In [21]:
def super_clean_date_final(row):
    url = str(row['URL'])
    
    # Bước 1: Tìm ngày tháng từ URL (vì URL của VietnamBiz chứa ngày rất chuẩn)
    match_date = re.search(r'(?:ngay|hom-nay)-(\d{1,4})', url)
    match_year = re.search(r'(202[3-5])', url)
    
    if match_date and match_year:
        date_str = match_date.group(1)
        year = match_year.group(1)
        try:
            if len(date_str) == 3: # 812 -> 08/12
                day, month = date_str[0].zfill(2), date_str[1:].zfill(2)
            elif len(date_str) == 4: # 1012 -> 10/12
                day, month = date_str[:2], date_str[2:]
            elif len(date_str) == 2: # 81 -> 08/01
                day, month = date_str[0].zfill(2), date_str[1].zfill(2)
            else:
                return row['Ngaydang'] # Trả về ngày gốc nếu ko khớp format
            return f"{year}-{month}-{day}"
        except:
            return row['Ngaydang']
            
    # Bước 2: Nếu URL không có ngày, trả về đúng ngày đang có ở cột Ngaydang
    return row['Ngaydang']

# --- THỰC THI ---
print("--- Đang xử lý file Excel (Giữ đủ 403 dòng) ---")

# 1. Đọc đúng file Excel của bạn
df = pd.read_excel('Gia_phan_fixed.xlsx')

# 2. KHÔNG DÙNG df = df[['URL']].copy() nữa vì sẽ làm mất cột Ngaydang gốc
# Ta giữ nguyên df để có data đối chiếu

# 3. Tạo cột Ngay_Chuan mà không làm mất dữ liệu gốc
df['Ngay_Chuan'] = df.apply(super_clean_date_final, axis=1)

# 4. Sắp xếp để kiểm tra (KHÔNG dropna)
# Chuyển sang datetime chỉ để sort, sau đó giữ nguyên format string cho sạch
df['Ngay_DT'] = pd.to_datetime(df['Ngay_Chuan'], errors='coerce')
df = df.sort_values(by='Ngay_DT', ascending=False)

# 5. Chỉ lấy các cột cần thiết để lưu, đảm bảo vẫn còn URL và Ngay_Chuan
df_final = df[['Ngay_Chuan', 'URL']]

# 6. Xuất file
df_final.to_excel('Gia_phan_Chuan_Full_403.xlsx', index=False)

print(f"✅ Hoàn thành! Số dòng: {len(df_final)}")
print(df_final.head(10))

--- Đang xử lý file Excel (Giữ đủ 403 dòng) ---
✅ Hoàn thành! Số dòng: 403
     Ngay_Chuan                                                URL
402  2025-12-19  https://vietnambiz.vn/gia-phan-bon-ngay-1912-p...
401  2025-12-18  https://vietnambiz.vn/gia-phan-bon-ngay-1812-t...
400  2025-12-17  https://vietnambiz.vn/gia-phan-bon-ngay-1712-t...
399  2025-12-16  https://vietnambiz.vn/gia-phan-bon-ngay-1612-p...
398  2025-12-15  https://vietnambiz.vn/gia-phan-bon-ngay-1512-b...
397  2025-12-12  https://vietnambiz.vn/gia-phan-bon-ngay-1212-d...
396  2025-12-11  https://vietnambiz.vn/gia-phan-bon-ngay-1112-k...
395  2025-12-10  https://vietnambiz.vn/gia-phan-bon-ngay-1012-t...
394  2025-12-09  https://vietnambiz.vn/gia-phan-bon-ngay-912-ph...
393  2025-12-08  https://vietnambiz.vn/gia-phan-bon-ngay-812-ph...


In [29]:
class FertilizerMegaScanner:
    # Mở rộng bộ từ khóa bắt giá
    KEYWORDS = {
        'NPK 16-16-8': [r'16[-–—\s]*16[-–—\s]*8', r'NPK\s*16\.16\.8'],
        'Kali': [r'Kali'],
        'Urê Cà Mau': [r'Cà\s*Mau', r'Đạm\s*Cà\s*Mau', r'Lạnh\s*Cà\s*Mau']
    }
    
    # Mở rộng bộ từ khóa khu vực để không sót Miền Tây
    WEST_KEYWORDS = ["Miền Tây", "Tây Nam Bộ", "ĐBSCL", "Long An", "Tiền Giang", "Đồng Tháp", "Cần Thơ", "An Giang"]

    @staticmethod
    def extract_price_ultra(keyword, text):
        # Tìm giá sau từ khóa, bỏ qua các ký tự gây nhiễu
        pattern = rf"{keyword}(?:[^0-9]*?)(\d{{1,3}}\.?\d{{3}})(?:\s*[-–—]\s*(\d{{1,3}}\.?\d{{3}}))?"
        match = re.search(pattern, text, re.IGNORECASE)
        
        if match:
            p1 = match.group(1).replace('.', '')
            
            # --- XỬ LÝ SỐ HIỆU 16168 ---
            if p1 == "16168":
                remaining = text[match.end():]
                # Tìm con số tiếp theo có định dạng giá tiền (ví dụ 750.000)
                next_p = re.search(r'(\d{1,3}\.\d{3})', remaining)
                if next_p:
                    p1 = next_p.group(1).replace('.', '')
                else:
                    return None
            
            # Chỉ lấy giá trị từ 100.000 đến 2.000.000
            try:
                val = int(p1)
                if 100000 <= val <= 2000000:
                    p2 = match.group(2).replace('.', '') if match.group(2) else None
                    return f"{p1}-{p2}" if p2 else p1
            except:
                pass
        return None

In [30]:
def run_mega_crawl():
    # Đọc file Excel 403 dòng
    file_path = 'Gia_phan_fixed.xlsx'
    df = pd.read_excel(file_path)
    
    print(f"🚀 Đang quét siêu sâu cho {len(df)} dòng...")

    final_results = []
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

    for i in range(len(df)):
        row = df.iloc[i]
        url = str(row['URL']).strip()
        
        # Xử lý ngày an toàn
        try:
            raw_date = row['Ngaydang']
            clean_date = pd.to_datetime(raw_date, dayfirst=True, errors='coerce')
            date_str = clean_date.strftime('%Y-%m-%d') if pd.notnull(clean_date) else str(raw_date)
        except:
            date_str = "N/A"

        entry = {
            "id": i + 1,
            "ngay_dang": date_str,
            "url": url,
            "data": {"NPK 16-16-8": None, "Kali": None, "Urê Cà Mau": None}
        }

        if "http" in url:
            try:
                res = requests.get(url, headers=headers, timeout=10)
                if res.status_code == 200:
                    soup = BeautifulSoup(res.content, 'html.parser')
                    
                    # 1. Tìm trong các Table trước (Lọc theo khu vực)
                    tables = soup.find_all('table')
                    found_in_table = False
                    for table in tables:
                        t_text = table.get_text(separator=' ')
                        if any(k in t_text for k in FertilizerMegaScanner.WEST_KEYWORDS):
                            for item, aliases in FertilizerMegaScanner.KEYWORDS.items():
                                for alias in aliases:
                                    price = FertilizerMegaScanner.extract_price_ultra(alias, t_text)
                                    if price:
                                        entry["data"][item] = price
                                        found_in_table = True
                    
                    # 2. DỰ PHÒNG: Nếu table không có, quét toàn văn bản (Search rộng hơn)
                    if not found_in_table:
                        full_text = soup.get_text(separator=' ')
                        for item, aliases in FertilizerMegaScanner.KEYWORDS.items():
                            for alias in aliases:
                                price = FertilizerMegaScanner.extract_price_ultra(alias, full_text)
                                if price:
                                    entry["data"][item] = price
            except:
                pass
        
        final_results.append(entry)

    # Xuất JSON
    with open('ket_qua_sieu_quet_403.json', 'w', encoding='utf-8') as f:
        json.dump(final_results, f, ensure_ascii=False, indent=4)
    
    print(f"✅ Hoàn thành! Kiểm tra file 'ket_qua_sieu_quet_403.json'.")

if __name__ == "__main__":
    run_mega_crawl()

🚀 Đang quét siêu sâu cho 403 dòng...
✅ Hoàn thành! Kiểm tra file 'ket_qua_sieu_quet_403.json'.
